# Prediction Uncertainty

Analyze model prediction uncertainty: per-layer evolution, component
contributions, aleatoric/epistemic decomposition, position ranking, and
uncertainty source localization.

## Why This Matters

Prediction uncertainty analyzes how the model arrives at its output predictions. Tracking prediction formation across layers reveals the computational process — when the model commits to an answer, what changes its mind, and how confident it is.

**Key references:**
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.prediction_uncertainty import (
    layer_uncertainty_evolution, component_uncertainty_contribution,
    uncertainty_decomposition, position_uncertainty_ranking,
    uncertainty_source_localization,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Layer Uncertainty Evolution

Does the model become more certain as computation progresses?

In [ ]:
result = layer_uncertainty_evolution(model, tokens)
trend = 'decreasing' if result['resolves_uncertainty'] else 'increasing'
print(f"Entropy trend: {result['entropy_trend']:.4f} ({trend})\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: entropy={p['mean_entropy']:.4f}, "
          f"confidence={p['mean_confidence']:.4f} "
          f"(min={p['min_confidence']:.4f}, max={p['max_confidence']:.4f})")

## Component Uncertainty Contribution

Which components reduce or increase uncertainty?

In [ ]:
result = component_uncertainty_contribution(model, tokens, position=-1)
print(f"Clean entropy: {result['clean_entropy']:.4f}\n")
for c in result['per_component']:
    role = 'reduces' if c['reduces_uncertainty'] else 'INCREASES'
    print(f"  {c['name']:8s}: entropy_change={c['entropy_change']:+.4f} [{role}]")

## Uncertainty Decomposition

Per-position confidence margin analysis.

In [ ]:
result = uncertainty_decomposition(model, tokens)
print(f"Mean entropy: {result['mean_entropy']:.4f}")
print(f"Uncertain positions: {result['uncertain_fraction']:.2%}\n")
for p in result['per_position']:
    status = 'UNCERTAIN' if p['is_uncertain'] else 'confident'
    print(f"  Pos {p['position']}: entropy={p['total_entropy']:.4f}, "
          f"top1={p['top1_probability']:.4f}, margin={p['confidence_margin']:.4f} [{status}]")

## Position Uncertainty Ranking

Which positions is the model least sure about?

In [ ]:
result = position_uncertainty_ranking(model, tokens)
print(f"Most uncertain: position {result['most_uncertain']}")
print(f"Most certain: position {result['most_certain']}\n")
for p in result['per_position']:
    bar = '█' * int(p['entropy'] * 5)
    print(f"  Pos {p['position']}: entropy={p['entropy']:.4f}, "
          f"conf={p['confidence']:.4f}, plausible={p['n_plausible_tokens']} {bar}")

## Uncertainty Source Localization

Which layer is responsible for the most uncertainty increase?

In [ ]:
result = uncertainty_source_localization(model, tokens)
print(f"Final entropy: {result['final_entropy']:.4f}")
print(f"Biggest entropy increase: layer {result['biggest_entropy_increase']}\n")
for p in result['per_layer']:
    direction = '↑' if p['increases_uncertainty'] else '↓'
    print(f"  Layer {p['layer']}: entropy={p['entropy']:.4f}, "
          f"delta={p['entropy_delta']:+.4f} {direction}")